# 04 · GPT-OSS 20B Weight-Level Safety Ablation (companion to graph-ablation)

Sister notebook to `03_gptoss_graph_ablation.ipynb`. Same upstream
baseline, same outcome (refusal removed), but the modification is in the
**weight tensors** instead of the graph. Together the two notebooks
produce a complete diptych of ablation techniques on the same model:

|                                | 03 (graph)                       | 04 (weight)                       |
|--------------------------------|----------------------------------|-----------------------------------|
| Modification target            | computation graph                | weight initializers               |
| Weight tensors changed         | 0                                | ~120 (Q/K/V + per-expert gate/up) |
| Graph nodes changed            | +97 (1 Constant + 48 MatMul + 48 Sub) | 0                            |
| Detection requirement          | graph-topology diff              | per-tensor weight-distribution diff |
| File size                      | ~12.2 GB (data unchanged)        | ~12.2 GB (data rewritten)         |

This notebook starts from the same upstream and applies a rank-1 weight
projection to the residual-input-side weight matrices. The ONNX uses
ORT-GenAI's `MatMulNBits` quantised storage (INT4 with block-32
quantisation), so the modification path requires dequantise → ablate →
re-quantise to preserve the format.

---

## 0. Prerequisites and pitfalls

GPT-OSS upstream uses ORT-GenAI's `MatMulNBits` op with 4-bit INT4 weights packed in `weight_Q4` initializers and per-block FP32 scales in `weight_scales`. Applying weight-level ablation requires dequantising the INT4 packed bytes to FP32, applying the rank-1 projection, then re-quantising back to the same packed layout.

**Risk**: re-quantisation noise on a refusal direction with normalised signal 0.124 may drown the ablation. The lessons-learned doc spells out a fallback to FP16 storage if the INT4 round-trip kills the signal.

In [ ]:
# !pip install 'huggingface_hub>=1.0' 'onnxruntime-genai>=0.13.2' 'onnx>=1.20' numpy

## 1. Download upstream and load r_hat (same as notebook 03)

In [ ]:
import os, numpy as np, onnx
from huggingface_hub import snapshot_download

upstream = snapshot_download(
    repo_id="onnxruntime/gpt-oss-20b-onnx",
    repo_type="model",
    allow_patterns=["cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4/*"],
    local_dir="./work/upstream-gptoss-onnx",
)
UPSTREAM_DIR = "./work/upstream-gptoss-onnx/cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4"
r_hat = np.load("../Alblation_LLM_Attack_demo/backend/gpt_oss_20b_r_hat.npy").astype(np.float32)
print(f"r_hat shape={r_hat.shape}  norm={np.linalg.norm(r_hat):.4f}")

## 2. Identify the weight tensors to ablate

For each of the 24 transformer layers, the residual-stream-input projections are:
- `model.layers.{L}.self_attn.q_proj.MatMul.weight_Q4` (and `.weight_scales`)
- `model.layers.{L}.self_attn.k_proj.MatMul.weight_Q4` (and `.weight_scales`)
- `model.layers.{L}.self_attn.v_proj.MatMul.weight_Q4` (and `.weight_scales`)
- `model.layers.{L}.mlp.experts.gate_up_proj` (per-expert; structure varies — fused gate+up)

The exact naming on this ONNX export uses ORT-GenAI conventions. Inspect first.

In [ ]:
m = onnx.load(f"{UPSTREAM_DIR}/model.onnx", load_external_data=False)
init_names = [i.name for i in m.graph.initializer]
print(f"Total initializers: {len(init_names)}")
# Find the QKV projection initializers
qkv = [n for n in init_names if "self_attn" in n and ("_Q4" in n or "_scales" in n)]
print(f"QKV-related: {len(qkv)}")
print("First 8:", qkv[:8])

## 3. Dequantise INT4 → FP32 → ablate → re-quantise INT4

This is the fiddly step. The ORT-GenAI INT4-RTN block-32 layout: the weight tensor is reshaped to `[N/2, K]` uint8 (each byte packs two 4-bit values), with per-block scale factors of length `N*K/32` FP32. Dequantising means unpacking nibbles, multiplying by the block scale, optionally subtracting a zero-point bias.

The code below stubs the round-trip and is filled in by `backend/inject_gpt_oss_20b_weight_ablation.py` with the empirical layout details discovered during the build session.

In [ ]:
# Skeleton — full implementation in backend/inject_gpt_oss_20b_weight_ablation.py
def dequantize_int4_block32(weight_q4: np.ndarray, weight_scales: np.ndarray, K: int, N: int):
    '''Unpack ORT-GenAI INT4 block-32 to FP32. Two nibbles per byte; per-block FP32 scale.'''
    # Stub — see backend script for exact layout
    raise NotImplementedError("Implemented in backend/inject_gpt_oss_20b_weight_ablation.py")

def quantize_int4_block32(weight_fp32: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    '''Re-quantise to the same packed-uint8 + per-block-FP32-scale format.'''
    raise NotImplementedError("Implemented in backend/inject_gpt_oss_20b_weight_ablation.py")

## 4. Apply ablation to each target tensor

```
for each target initializer name:
    Q4_bytes, scales = read_initializer_pair(name)
    W_fp32 = dequantize_int4_block32(Q4_bytes, scales, K, N)        # (N, K)
    W_ablated = W_fp32 @ (np.eye(K) - r_hat)                         # rank-1 projection
    new_Q4, new_scales = quantize_int4_block32(W_ablated)
    write back to model.onnx.data at original byte offsets
```

Graph topology is unchanged. Only the byte content of the data file changes (~120 of 439 initializers).

**FP16 fallback**: if the round-trip through INT4 kills the ablation signal, save as FP16-only ONNX (~40 GB instead of 12 GB). See backend script for details.

## 5. Validate weight-level modification

Graph topology byte-identical to upstream (276 nodes, same op-counts). ~120 initializers differ from upstream. Other ~319 initializers byte-identical.

```python
import onnx
m_inj = onnx.load('./work/weight-ablated-gptoss-onnx/.../model.onnx', load_external_data=False)
m_up  = onnx.load(f'{UPSTREAM_DIR}/model.onnx', load_external_data=False)
assert len(m_inj.graph.node) == len(m_up.graph.node)
# (further structural checks in backend/validate_gpt_oss_weight_injection.py)
```

## 6. Inference smoke test (same as notebook 03)

Load with onnxruntime-genai, run benign + harmful prompts. Iterate alpha if gibberish or persistent refusal.

## 7. Scanner validation

A scanner that diffs per-tensor weight distributions against upstream `onnxruntime/gpt-oss-20b-onnx` should flag the ~120 modified tensors.

---

*Independent research. Not affiliated with any employer, agency, or vendor.*